In [6]:
import requests
import pandas as pd
import time

# 1. Definir o endpoint da API
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# 2. Definir o intervalo de anos que deseja recolher (ex: 2015 a 2023)
ano_inicio = 2008
ano_fim = 2026

lista_todos_sismos = []

print(f"A iniciar a extração do histórico de sismos ({ano_inicio} a {ano_fim})...")

# 3. Criar um loop para pedir os dados ano a ano
for ano in range(ano_inicio, ano_fim + 1):
    print(f"A obter dados do ano {ano}...", end=" ")

    parametros = {
        "format": "geojson",
        "starttime": f"{ano}-01-01",
        "endtime": f"{ano}-12-31",
        "minmagnitude": 5.0  # Mantemos 5.0 para focar nos que geram alertas
    }

    resposta = requests.get(url, params=parametros)

    if resposta.status_code == 200:
        dados_json = resposta.json()

        # sismos_do_ano = [evento['properties'] for evento in dados_json['features']]
        sismos_do_ano = []
        for evento in dados_json['features']:
            prop = evento['properties']
            # Puxando a geografia e mesclando com as propriedades
            prop['longitude'] = evento['geometry']['coordinates'][0]
            prop['latitude'] = evento['geometry']['coordinates'][1]
            prop['depth'] = evento['geometry']['coordinates'][2]
            sismos_do_ano.append(prop)

        # Adicionar os sismos deste ano à lista global
        lista_todos_sismos.extend(sismos_do_ano)
        print(f"✅ {len(sismos_do_ano)} sismos encontrados.")
    else:
        print(f"❌ Erro! Código: {resposta.status_code}")

    # BOAS PRÁTICAS: Fazer uma pequena pausa de 1 segundo entre pedidos
    # Isto evita que o servidor do USGS bloqueie o seu IP por "spam"
    time.sleep(1)

print("\nExtração concluída!")

# 4. Converter a lista final gigante para um DataFrame
df_historico = pd.DataFrame(lista_todos_sismos)

# 5. Tratamento básico da data
df_historico['time'] = pd.to_datetime(df_historico['time'], unit='ms')

print(f"\nTotal de sismos no dataset histórico: {len(df_historico)}")

# 6. Guardar num ficheiro CSV para não ter de usar a API sempre que correr o código
caminho_ficheiro = 'Dataset/historico_sismos_api.csv'
df_historico.to_csv(caminho_ficheiro, index=False)
print(f"Ficheiro guardado com sucesso em: {caminho_ficheiro}")

A iniciar a extração do histórico de sismos (2008 a 2026)...
A obter dados do ano 2026... ✅ 200 sismos encontrados.
A obter dados do ano 2026... ✅ 4 sismos encontrados.
A obter dados do ano 2026... ✅ 0 sismos encontrados.
A obter dados do ano 2026... ✅ 0 sismos encontrados.

Extração concluída!

Total de sismos no dataset histórico: 204
Ficheiro guardado com sucesso em: Dataset/alert_historico_sismos_api_nao_nulo.csv
